In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

from pprint import pprint
import pickle

import pandas as pd

from functools import partial

import numpy as np
from sklearn.preprocessing import scale
# import sklearn.metrics as metrics
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
# import matplotlib.animation as ani

import mlgrad.models as models
import mlgrad.regr as regr
import mlgrad.cls as cls
import mlgrad.pca as pca
import mlgrad.plots as plots
import mlgrad.funcs as funcs
import mlgrad.funcs2 as funcs2
import mlgrad.loss as loss
import mlgrad.gd as gd
import mlgrad.smooth as smooth
import mlgrad.inventory as inventory
from mlgrad.af import averaging_function
import mlgrad.avragg as avragg

# import rampy
# import pybaselines
import random

from irsa import io
from irsa.preprocess import despike, utils
import irsa.widgets as widgets
import irsa.spectra as spectra
import irsa.io as io

import scipy.special as special

import copy

import ipywidgets
from IPython.display import display

import sys
print(sys.version)

# %matplotlib widget

3.14.6 (main, Jun 10 2026, 18:54:31) [GCC 15.2.0]


In [ ]:
def plot_one_param(xs, ps, p_threshold=1.0e-4):
    n = len(ps)
    ps_p = ps.copy()
    ps_m = ps.copy()
    ps_p[ps < 0] = 0
    ps_m[ps > 0] = 0
    plt.vlines(xs, ps_m, ps_p)
    
def plot_params(lfm, p_threshold=1.0e-4):
    ax = plt.twinx(plt.gca())
    # ax.plot((0,0), (mod.n_param,0), ls='--', lw=0.5, color='b')
    n = lfm.n_input
    mask = np.zeros(n, np.uint8)
    for mod in lfm.models:
        param1 = np.asarray(mod.param[1:])
        # print(param1[0])
        param_p = param1.copy()
        param_m = param1.copy()
        param_p[param1 < 0] = 0
        param_m[param1 > 0] = 0
        mask = mask | (abs(param1) > p_threshold)
        ax.vlines(range(mod.n_input), param_m, param_p, color='k')
    ax.minorticks_on()
    s = mask.astype("i").sum()
    plt.title(f'#(abs(param) > {p_threshold}) = {s}')

In [ ]:
path = f"/home/intellimath/YD/repos/spectra/21_SU"
options = {}
dd = {}
# widgets.load_spectra(path, dd, options)

In [ ]:
path = f"/home/intellimath/YD/repos/spectra/21_SU"
dd = io.load_spectra(path, { #"вид_бактерий":["SA", "SS", "SH"], 
                            "номер_цикла":["21"], 
                            "лизис":"термический",
                            # "номер_эксперимента_в_цикле":["2", "4"],
                            "отсечки_по_молекулярной_массе": ["super"],
                            
                           }, verbose=False)

In [ ]:
for key, sp in dd.spectra.items():
    print(key, len(sp.y))

In [ ]:
df = pd.read_csv("spectra_20_tab1.csv", sep="\t")
cls_col = df["cls"]
cls_name_col = cls_col.apply(lambda x: x.split("/")[0])
cls_nn_col = cls_col.apply(lambda x: x.split("/")[1])
df.__delitem__("cls")
df["cls_name"] = cls_name_col
df["cls_nn"] = cls_nn_col
id1_array = df["id1"].values
id2_array = df["id2"].values
id_to_cls = {int(id):name.upper() for id,name in zip(id1_array, cls_name_col.values)}
id_to_cls

In [ ]:
dd1 = spectra.SpectraCollection()
for key in dd.spectra.keys():
    keys = key.split("_")
    id0 = keys[0]
    id = int(id0[2:])
    keys[1] = id0
    keys[0] = id_to_cls[id].upper()
    new_key = "_".join(keys)
    sp = dd.spectra[key]
    sp.attrs["key"] = new_key
    sp.attrs["вид_бактерий"] = keys[0]
    dd1[new_key] = sp
dd = dd1
del dd1

In [ ]:
# dd.apply(lambda x,y: np.log(y))

In [ ]:
dd.crop(400,1700)
dd.align_bottom()
# dd.scale_by_mean()

In [ ]:
dd.drop_zeros_spectrum()
dd.drop_notfinite()
dd.drop_empty()
dd.isfinite()

In [ ]:
for key, sp in dd.spectra.items():
    print(key, sp.y[0].shape)

In [ ]:
dd.despike(2.0)

In [ ]:
dd.denoise_svd()

In [ ]:
dd.filter()

In [ ]:
dd.scale_by_mean()

In [ ]:
dd.robust_averaging()

In [ ]:
# dd.plot_spectra(fract=True)

In [ ]:
# dd.plot_pca_spectra(10)

In [ ]:
# dd.plot_spectra(smoothing=True, kind="svd", method="energy", threshold=0.9975, transpose=True)

In [ ]:
# dd.flatten()
# dd.save("/home/intellimath/work/spectra/20_term2", "flatten")

In [ ]:
# dd.select_baselines(tau2=1.0e4, #tau1=-1.0e-3,
#                       func=funcs.RStep(0.01, 1.0),
#                       # func2=funcs.RStep(0.001, 0.001),
#                       # func2_e=inventory.relative_abs_max,
# )

In [ ]:
dd.select_baselines(tau2=1.0e3, #tau1=-1.0e-3,
                      func=funcs.RStep(0.0, 0.001),
                      # func2=funcs.RStep(0.001, 0.001),
                      func2_e=inventory.relative_abs_max,
)

In [ ]:
dd.subtract_baselines(tau2=1.0e3, #tau1=-1.0e-3,
                      func=funcs.RStep(0.0, 0.001),
                      func2_e=inventory.relative_abs_max,
                      verbose=False, n_jobs=8
                     )

In [ ]:
# dd.apply(lambda x,y: np.exp(y))

In [ ]:
dd.scale_by_mean()

In [ ]:
dd.filter()

In [ ]:
dd.plot_spectra(fract=True)

In [ ]:
dd.save("/home/intellimath/work/spectra/21_su", "rm_bl")

In [ ]:
dd = spectra.SpectraCollection()
dd.load("/home/intellimath/work/spectra/21_su", "rm_bl")

In [ ]:
dd.plot_spectra(False)

In [ ]:
with open("pca_20_term2.pickle", mode="rb") as ff:
    c, As, Ls, U_SA, U_SS, U_SH = pickle.load(ff)

In [ ]:
@ipywidgets.interact(key=list(dd.keys()))
def plot_pca(key):
    sp = dd.spectra[key]
    Ys = sp.y
    # U = (Ys - c) @ As.T
    U = Ys @ As.T
    plt.figure(figsize=(13,3))
    for i in range(4):
        plt.subplot(1,4,i+1)
        plt.scatter(U_SA[:,i], U_SA[:,i+1], s=25, c='r', edgecolors='k', linewidth=0.5, label="SA")
        plt.scatter(U_SH[:,i], U_SH[:,i+1], s=25, c='g', edgecolors='k', linewidth=0.5, label="SH")
        plt.scatter(U_SS[:,i], U_SS[:,i+1], s=25, c='b', edgecolors='k', linewidth=0.5, label="SS")
        plt.scatter(U[:,i], U[:,i+1], s=25, c='m', edgecolors='k', linewidth=0.5, label="Test")
        plt.xlabel(str(i+1))
        plt.ylabel(str(i+2))
        # plt.xticks([])
        # plt.yticks([])
        plt.legend()
    plt.tight_layout()
    plt.legend()
    plt.show()



In [ ]:
import mpltern
print(len(ys))

ax = plt.subplot(projection="ternary")
Z1 = lfm1.evaluate(ys)
Z2 = lfm2.evaluate(ys)
Z3 = lfm3.evaluate(ys)
Y1 = (1 + (Z1 - Z3) / 2) / 2
Y2 = (1 + (Z2 - Z1) / 2) / 2
Y3 = (1 + (Z3 - Z2) / 2) / 2
Y1[Y1<0] = 0
Y2[Y2<0] = 0
Y3[Y3<0] = 0
print(((Z1>0) & (Z3<0)).sum())
print(((Z2>0) & (Z1<0)).sum())
print(((Z2>0) & (Z3<0)).sum())
# print(Y1)
# print(Y2)
# print(Y3)
C = [(y1,y2,y3) for y1,y2,y3 in zip(Y1,Y2,Y3)]
# cax = ax.inset_axes([1.05, 0.1, 0.05, 0.9], transform=ax.transAxes)
cs = ax.scatter(Y1, Y2, Y3, c=zs, s=9)
plt.colorbar(cs)
ax.axline(
    [0.5, 0.0, 0.0],
    [0.5, 0.5, 0.5])
ax.axline(
    [0.0, 0.5, 0.0],
    [0.5, 0.5, 0.5])
ax.axline(
    [0.0, 0.0, 0.5],
    [0.5, 0.5, 0.5])
plt.show()

zz1 = Y1
zz2 = Y2 * np.exp(2j/3 * np.pi)
zz3 = Y3 * np.exp(-2j/3 * np.pi)
C = np.concatenate((Zs_SS, 2*Zs_SH, 3*Zs_SA))
plt.figure(figsize=(8,6))
cs = plt.scatter(np.real(zz1+zz2+zz3), np.imag(zz1+zz2+zz3), s=9, c=C)
plt.colorbar(cs)
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression

XX = np.c_[np.real(zz1+zz2+zz3), np.imag(zz1+zz2+zz3)]
YY = np.concatenate((Zs_SS-1, Zs_SH, 2*Zs_SA))

lrg = LogisticRegression()
lrg.fit(XX, YY)
print(accuracy_score(YY, lrg.predict(XX)))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
def plot_confusion_matrix(Y, Yp, labels): #, Y_test, Z, Z_test):
    cm = confusion_matrix(Y, Yp, normalize="true")
    plt.figure(figsize=(4,3))
    ax = sns.heatmap(cm, annot=True, cmap='YlGnBu', fmt='0.2f',
                     xticklabels=labels, yticklabels=labels)
    ax.xaxis.tick_top()
    plt.xticks(rotation=90) 
    plt.show()


In [ ]:
plot_confusion_matrix(YY, lrg.predict(XX), labels=["SS", "SH", "SA"])